# Distributed Training with Ray Train

This notebook demonstrates how to scale model training across multiple GPUs using Ray Train. You'll learn how to:

- Distribute PyTorch training with minimal code changes
- Train XGBoost models on large datasets across a cluster
- Configure scaling and fault tolerance
- Manage checkpoints for long-running jobs

## Prerequisites

- 2-8 GPUs (for PyTorch)
- Python 3.9+

## 1. Installation

In [ ]:
!pip install -q "ray[train]" torch torchvision xgboost-ray scikit-learn pandas matplotlib

## 2. Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")

from utils import (
    print_gpu_status,
    get_recommended_workers,
    init_ray,
    shutdown_ray,
    ClusterMode,
    get_scaling_config,
    get_run_config,
    plot_training_curves,
)

In [ ]:
print_gpu_status()

NUM_WORKERS = get_recommended_workers()
print(f"\nRecommended workers: {NUM_WORKERS}")

In [ ]:
# Configuration
CLUSTER_MODE = ClusterMode.LOCAL  # Change to ClusterMode.ANYSCALE for Anyscale

# Storage paths
STORAGE_PATH = "./runs/distributed_training"

In [ ]:
init_ray(mode=CLUSTER_MODE)

---

# Part 1: Distributed PyTorch Training

Train a ResNet model on CIFAR-10 across multiple GPUs.

## 1.1 Define Training Function

The training function runs on each worker. Ray Train handles:
- Distributed data parallel (DDP) setup
- Data sharding across workers
- Gradient synchronization

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import ray
from ray import train
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig, CheckpointConfig


def train_func_pytorch(config):
    """
    PyTorch training function that runs on each worker.
    
    Ray Train automatically:
    - Sets up distributed process group
    - Wraps model in DistributedDataParallel
    - Shards data across workers
    """
    # Hyperparameters from config
    batch_size = config.get("batch_size", 64)
    epochs = config.get("epochs", 5)
    lr = config.get("lr", 0.001)
    
    # Data transforms
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # Load datasets
    train_dataset = datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform_train
    )
    test_dataset = datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform_test
    )
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Prepare data loaders for distributed training
    # This shards data across workers automatically
    train_loader = train.torch.prepare_data_loader(train_loader)
    test_loader = train.torch.prepare_data_loader(test_loader)
    
    # Create model
    model = models.resnet18(weights=None, num_classes=10)
    
    # Prepare model for distributed training
    # This wraps in DDP and moves to correct device
    model = train.torch.prepare_model(model)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
        
        train_acc = 100.0 * correct / total
        avg_loss = train_loss / len(train_loader)
        
        # Evaluation
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for inputs, targets in test_loader:
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                test_total += targets.size(0)
                test_correct += predicted.eq(targets).sum().item()
        
        test_acc = 100.0 * test_correct / test_total
        
        # Report metrics to Ray Train
        train.report(
            metrics={
                "loss": avg_loss,
                "train_accuracy": train_acc,
                "test_accuracy": test_acc,
                "epoch": epoch,
            }
        )
        
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, "
              f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

## 1.2 Configure and Run Training

In [ ]:
# Training configuration
train_config = {
    "batch_size": 64,
    "epochs": 5,
    "lr": 0.001,
}

# Scaling configuration
scaling_config = get_scaling_config(
    num_workers=NUM_WORKERS,
    use_gpu=True,
)

# Run configuration with checkpointing
run_config = get_run_config(
    name="pytorch-cifar10",
    storage_path=STORAGE_PATH,
)

print(f"Training with {NUM_WORKERS} workers")
print(f"Config: {train_config}")

In [ ]:
# Create trainer
pytorch_trainer = TorchTrainer(
    train_loop_per_worker=train_func_pytorch,
    train_loop_config=train_config,
    scaling_config=scaling_config,
    run_config=run_config,
)

print("TorchTrainer created successfully")

In [ ]:
# Run distributed training
print("Starting distributed PyTorch training...")
print("="*50)

pytorch_result = pytorch_trainer.fit()

print("="*50)
print("Training completed!")
print(f"Final test accuracy: {pytorch_result.metrics.get('test_accuracy', 'N/A'):.2f}%")

In [ ]:
# Visualize training progress
# Note: In practice, use TensorBoard or Weights & Biases for monitoring
print("\nTraining metrics from result:")
print(f"  Final loss: {pytorch_result.metrics.get('loss', 'N/A'):.4f}")
print(f"  Final train accuracy: {pytorch_result.metrics.get('train_accuracy', 'N/A'):.2f}%")
print(f"  Final test accuracy: {pytorch_result.metrics.get('test_accuracy', 'N/A'):.2f}%")

---

# Part 2: Distributed XGBoost Training

Train XGBoost models on tabular data across multiple workers.

In [ ]:
from ray.train.xgboost import XGBoostTrainer
from sklearn.datasets import make_classification
import pandas as pd

## 2.1 Prepare Dataset

In [ ]:
# Generate synthetic classification dataset
# In production, load from Parquet, CSV, or database
X, y = make_classification(
    n_samples=100000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=3,
    random_state=42,
)

# Create DataFrame
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["label"] = y

print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

In [ ]:
# Convert to Ray Dataset for distributed loading
ray_dataset = ray.data.from_pandas(df)

# Split into train and validation
train_ds, val_ds = ray_dataset.train_test_split(test_size=0.2)

print(f"Training samples: {train_ds.count()}")
print(f"Validation samples: {val_ds.count()}")

## 2.2 Configure XGBoost Trainer

In [ ]:
# XGBoost parameters
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "error"],
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

# Scaling config - XGBoost uses CPU workers by default
# For GPU XGBoost, set tree_method="gpu_hist" and use_gpu=True
xgb_scaling_config = ScalingConfig(
    num_workers=4,  # Number of distributed workers
    use_gpu=False,  # Set True for GPU training
)

# Run config
xgb_run_config = get_run_config(
    name="xgboost-classification",
    storage_path=STORAGE_PATH,
)

print("XGBoost configuration:")
for k, v in xgb_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Create XGBoost trainer
xgb_trainer = XGBoostTrainer(
    scaling_config=xgb_scaling_config,
    run_config=xgb_run_config,
    label_column="label",
    num_boost_round=100,
    params=xgb_params,
    datasets={"train": train_ds, "valid": val_ds},
)

print("XGBoostTrainer created successfully")

In [ ]:
# Run distributed XGBoost training
print("Starting distributed XGBoost training...")
print("="*50)

xgb_result = xgb_trainer.fit()

print("="*50)
print("Training completed!")
print(f"Best validation error: {xgb_result.metrics.get('valid-error', 'N/A'):.4f}")

In [ ]:
# Print XGBoost results
print("\nXGBoost Training Results:")
print(f"  Validation Log Loss: {xgb_result.metrics.get('valid-logloss', 'N/A'):.4f}")
print(f"  Validation Error: {xgb_result.metrics.get('valid-error', 'N/A'):.4f}")
print(f"  Checkpoint: {xgb_result.checkpoint}")

## 2.3 Load and Use Trained Model

In [ ]:
import xgboost as xgb
from ray.train import Checkpoint

# Load model from checkpoint
if xgb_result.checkpoint:
    with xgb_result.checkpoint.as_directory() as checkpoint_dir:
        model_path = os.path.join(checkpoint_dir, "model.ubj")
        if os.path.exists(model_path):
            booster = xgb.Booster()
            booster.load_model(model_path)
            print(f"Model loaded from {model_path}")
            
            # Feature importance
            importance = booster.get_score(importance_type="gain")
            print("\nTop 5 Features by Importance:")
            sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]
            for feat, score in sorted_importance:
                print(f"  {feat}: {score:.4f}")

---

# Part 3: Fault Tolerance and Checkpointing

Ray Train provides automatic fault tolerance for long-running jobs.

In [ ]:
from ray.train import Checkpoint


def train_with_checkpointing(config):
    """
    Training function with explicit checkpointing.
    
    Checkpoints allow:
    - Resume training after failures
    - Save model at each epoch
    - Enable preemption recovery on spot instances
    """
    import tempfile
    
    epochs = config.get("epochs", 5)
    start_epoch = 0
    
    # Check for existing checkpoint to resume from
    checkpoint = train.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            # Load model state, optimizer state, epoch, etc.
            checkpoint_path = os.path.join(checkpoint_dir, "checkpoint.pt")
            if os.path.exists(checkpoint_path):
                state = torch.load(checkpoint_path)
                start_epoch = state["epoch"] + 1
                print(f"Resuming from epoch {start_epoch}")
    
    # Training loop
    for epoch in range(start_epoch, epochs):
        # ... training code ...
        loss = 0.5 / (epoch + 1)  # Dummy loss
        
        # Save checkpoint
        with tempfile.TemporaryDirectory() as temp_dir:
            checkpoint_path = os.path.join(temp_dir, "checkpoint.pt")
            torch.save({"epoch": epoch, "loss": loss}, checkpoint_path)
            
            # Report with checkpoint
            train.report(
                metrics={"loss": loss, "epoch": epoch},
                checkpoint=Checkpoint.from_directory(temp_dir),
            )

print("Checkpointing example defined. See function code for details.")

## Cleanup

In [ ]:
shutdown_ray()
print("Ray cluster shutdown complete")

## Key Takeaways

1. **Minimal code changes**: Ray Train wraps existing training code with a few `prepare_*` calls
2. **Automatic DDP**: No manual process group setup needed
3. **Data sharding**: `prepare_data_loader` handles distributed data automatically
4. **XGBoost native support**: Built-in `XGBoostTrainer` for tree-based models
5. **Fault tolerance**: Checkpointing enables resume after failures

## Next Steps

- **Hyperparameter tuning**: See `hyperparameter_tuning.ipynb` for Ray Tune integration
- **Larger models**: Use DeepSpeed integration for models that don't fit on one GPU
- **Production**: Deploy on Anyscale with spot instance recovery

## Resources

- [Ray Train Documentation](https://docs.ray.io/en/latest/train/train.html)
- [PyTorch Distributed Training](https://docs.ray.io/en/latest/train/getting-started-pytorch.html)
- [XGBoost with Ray](https://docs.ray.io/en/latest/train/getting-started-xgboost.html)
- [Fault Tolerance](https://docs.ray.io/en/latest/train/user-guides/fault-tolerance.html)